In [37]:
import numpy as np
import pandas as pd

In [38]:
df = pd.read_csv('Electric_Vehicle_Population_Data.csv')
print(df.shape)
print(df.dtypes)

(279780, 16)
VIN (1-10)                                            object
County                                                object
City                                                  object
State                                                 object
Postal Code                                          float64
Model Year                                             int64
Make                                                  object
Model                                                 object
Electric Vehicle Type                                 object
Clean Alternative Fuel Vehicle (CAFV) Eligibility     object
Electric Range                                       float64
Legislative District                                 float64
DOL Vehicle ID                                         int64
Vehicle Location                                      object
Electric Utility                                      object
2020 Census Tract                                    float64
dtype: obje

**1. Xử lý cột**

In [39]:
df = df.rename(columns={
    'VIN (1-10)': 'VIN',
    'Electric Vehicle Type': 'EV Type',
    'Clean Alternative Fuel Vehicle (CAFV) Eligibility': 'CAFV Eligibility',
    '2020 Census Tract': 'Census Tract'
})

print(df.columns.tolist())

['VIN', 'County', 'City', 'State', 'Postal Code', 'Model Year', 'Make', 'Model', 'EV Type', 'CAFV Eligibility', 'Electric Range', 'Legislative District', 'DOL Vehicle ID', 'Vehicle Location', 'Electric Utility', 'Census Tract']


In [40]:
# EV Type
df['EV Type'] = df['EV Type'].map({
    'Battery Electric Vehicle (BEV)': 'BEV',
    'Plug-in Hybrid Electric Vehicle (PHEV)': 'PHEV'
})

#CAFV Eligibility
cafv_mapping = {
    'Clean Alternative Fuel Vehicle Eligible': 'Eligible',
    'Not eligible due to low battery range': 'Not Eligible',
    'Eligibility unknown as battery range has not been researched': 'Unknown'}
df['CAFV Eligibility'] = df['CAFV Eligibility'].map(cafv_mapping)

In [41]:
# Tách giá trị của Vehicle Location
# Dùng regex tách ra longitude và latitude

df[['Longitude', 'Latitude']] = df['Vehicle Location'].str.extract(r'POINT \(([^ ]+) ([^ ]+)\)').astype(float)
df = df.drop(columns=['Vehicle Location'])

**2. Xử lý Missing Value**

In [42]:
df.isna().sum()

VIN                       0
County                   24
City                     24
State                     0
Postal Code              24
Model Year                0
Make                      0
Model                     0
EV Type                   0
CAFV Eligibility          0
Electric Range           11
Legislative District    700
DOL Vehicle ID            0
Electric Utility         24
Census Tract             24
Longitude               109
Latitude                109
dtype: int64

In [43]:
# Xem các dòng null của County
null_county = df[df['County'].isnull()]
print(f"Số dòng County null: {len(null_county)}")

# Kiểm tra các cột địa lý khác trong cùng những dòng đó
print("\nMissing values của các cột địa lý trong 24 dòng County null:")
print(null_county[['City', 'Postal Code', 'Electric Utility', 'Census Tract']].isnull().sum())

# --> Các biến GEO (County, City, State, Postal Code, Electric Utility, Census Tract) cùng null trên 24 dòng

Số dòng County null: 24

Missing values của các cột địa lý trong 24 dòng County null:
City                24
Postal Code         24
Electric Utility    24
Census Tract        24
dtype: int64


In [44]:
# Drop các feature có tỷ lệ Missing nhỏ
df = df.dropna(subset=['County', 'City', 'Postal Code', 'Electric Utility', 'Census Tract'])
df = df.dropna(subset=['Electric Range'])

# Kiểm tra kết quả
print(f"Shape sau khi xử lý: {df.shape}")

#print(df.isnull().sum())

Shape sau khi xử lý: (279745, 17)


In [45]:
df.columns

Index(['VIN', 'County', 'City', 'State', 'Postal Code', 'Model Year', 'Make',
       'Model', 'EV Type', 'CAFV Eligibility', 'Electric Range',
       'Legislative District', 'DOL Vehicle ID', 'Electric Utility',
       'Census Tract', 'Longitude', 'Latitude'],
      dtype='object')

In [47]:
df.to_csv('EV_Data.csv', index=False, encoding='utf-8-sig')